In [9]:
# Cell 1 - Setup: install NGBoost if needed, imports, paths
!pip install ngboost --quiet

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> project root
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from ngboost import NGBRegressor
from ngboost.distns import Normal
from ngboost.scores import LogScore

from models.utils import (
    chronological_split,
    select_enhanced_features,
    TARGET_REG,
    TARGET_CLASS,
    regression_report,
    classification_report,
    apply_feature_transforms,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

FEATURES = PROJECT_ROOT / "data" / "features"


In [10]:
# Cell 2 - Load feature matrix, chronological split, feature list
mat = pd.read_parquet(FEATURES / "feature_matrix_engineered_v2.parquet")
mat = apply_feature_transforms(mat)
train, val, test = chronological_split(mat)

features = select_enhanced_features(mat)
print(f"\n{len(features)} features selected")
print("First 10:", features[:10], "...")
print("Last 10: ", features[-10:])


  Split boundaries:
    Train: 2016-01-31 06:00:00+00:00 → 2022-12-31 23:00:00+00:00 (60,429 rows)
    Val:   2023-01-01 00:00:00+00:00 → 2023-12-31 23:00:00+00:00 (8,760 rows)
    Test:  2024-01-01 00:00:00+00:00 → 2024-12-31 06:00:00+00:00 (8,767 rows)

63 features selected
First 10: ['da_HB_HOUSTON', 'da_HB_HUBAVG', 'da_HB_NORTH', 'da_HB_SOUTH', 'da_HB_WEST', 'day_of_week', 'gdelt_article_volume', 'gdelt_norm', 'gdelt_tone', 'gdelt_tone_change_1d'] ...
Last 10:  ['stress_gas_spike', 'stress_heat_wave', 'stress_low_wind', 'stress_reactor_outage', 'stress_score', 'temp_max_across_zones', 'temp_min_across_zones', 'temp_range_across_zones', 'wind_mean_across_zones', 'wind_min_across_zones']


In [4]:
# Cell 3 - Shared model arrays for regression and classification
df_reg = mat.dropna(subset=[TARGET_REG])
tr_reg, vl_reg, te_reg = chronological_split(df_reg)

X_train_r = tr_reg[features].ffill().fillna(0);  y_train_r = tr_reg[TARGET_REG]
X_val_r   = vl_reg[features].ffill().fillna(0);  y_val_r   = vl_reg[TARGET_REG]
X_test_r  = te_reg[features].ffill().fillna(0);  y_test_r  = te_reg[TARGET_REG]

df_clf = mat.dropna(subset=[TARGET_CLASS])
tr_clf, vl_clf, te_clf = chronological_split(df_clf)

X_train_c = tr_clf[features].ffill().fillna(0);  y_train_c = tr_clf[TARGET_CLASS]
X_val_c   = vl_clf[features].ffill().fillna(0);  y_val_c   = vl_clf[TARGET_CLASS]
X_test_c  = te_clf[features].ffill().fillna(0);  y_test_c  = te_clf[TARGET_CLASS]

print(f"Regression arrays:     X_train={X_train_r.shape}, X_val={X_val_r.shape}, X_test={X_test_r.shape}")
print(f"Classification arrays: X_train={X_train_c.shape}, X_val={X_val_c.shape}, X_test={X_test_c.shape}")


  Split boundaries:
    Train: 2016-01-31 06:00:00+00:00 → 2022-12-31 23:00:00+00:00 (60,428 rows)
    Val:   2023-01-01 00:00:00+00:00 → 2023-12-31 23:00:00+00:00 (8,760 rows)
    Test:  2024-01-01 00:00:00+00:00 → 2024-12-31 06:00:00+00:00 (8,767 rows)
  Split boundaries:
    Train: 2016-01-31 06:00:00+00:00 → 2022-12-31 23:00:00+00:00 (60,429 rows)
    Val:   2023-01-01 00:00:00+00:00 → 2023-12-31 23:00:00+00:00 (8,760 rows)
    Test:  2024-01-01 00:00:00+00:00 → 2024-12-31 06:00:00+00:00 (8,767 rows)
Regression arrays:     X_train=(60428, 63), X_val=(8760, 63), X_test=(8767, 63)
Classification arrays: X_train=(60429, 63), X_val=(8760, 63), X_test=(8767, 63)


In [ ]:
# Cell 4 - NGBoost Regressor on log1p target (prevents σ overflow)
ngb = NGBRegressor(
    Dist=Normal,
    Score=LogScore,
    n_estimators=500,
    learning_rate=0.05,
    random_state=42,
    verbose=True,
)

y_train_log = np.log1p(np.clip(y_train_r.values, 0, None))

print("NGBoost training on log1p(price); this will take a few minutes...")
ngb.fit(X_train_r, y_train_log)

# NGBoost predicts (loc_log, scale_log) — Normal in LOG-DOLLAR space
test_dist = ngb.pred_dist(X_test_r)
test_mean_log = np.asarray(test_dist.loc)
test_std_log  = np.asarray(test_dist.scale)

# Invert for the $/MWh point prediction used in regression_report and
# the appended PR-AUC cell. expm1 is the exact inverse of log1p.
test_mean = np.expm1(test_mean_log)
test_std  = test_std_log   # log-space std; kept under same name for downstream code compatibility

print("\nTest-set metrics using predictive mean (inverted to $/MWh):")
reg_report = regression_report(
    y_test_r.values, test_mean,
    name="ngboost_normal_logscore_regressor_log1p test",
)

dist_summary = pd.DataFrame(
    {"mean_log": test_mean_log, "std_log": test_std_log, "mean_$": test_mean},
    index=te_reg.index,
)
display(dist_summary.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

In [ ]:
# Cell 5 - Spike probability in LOG SPACE (avoids the σ-overflow path).
# log-space threshold corresponding to $200/MWh:
SPIKE_THRESH_LOG = np.log1p(200.0)
safe_std_log = np.clip(test_std_log, 1e-6, None)
spike_proba = 1.0 - norm.cdf(SPIKE_THRESH_LOG, loc=test_mean_log, scale=safe_std_log)
spike_proba = np.clip(spike_proba, 0.0, 1.0)

spike_proba_s = pd.Series(spike_proba, index=te_reg.index, name="p_price_gt_200")
y_test_c_s = y_test_c.dropna().astype(int)
eval_idx = spike_proba_s.index.intersection(y_test_c_s.index)

print(f"Aligned spike-probability rows: {len(eval_idx):,} of {len(spike_proba_s):,} regression test rows")
print("\nTest metrics from NGBoost Gaussian spike probabilities:")
spike_report = classification_report(
    y_test_c_s.loc[eval_idx].values,
    spike_proba_s.loc[eval_idx].values,
    name="ngboost_gaussian_p_price_gt_200 test",
    threshold=0.5,
)

print("Compare this PR-AUC directly against the calibrated XGBoost classifier in notebooks/02_v2_models.ipynb.")
qs = np.quantile(spike_proba_s.loc[eval_idx], [0, 0.25, 0.5, 0.75, 0.95, 1.0])
print("\nNGBoost Gaussian spike-probability distribution on aligned test rows:")
print(f"  min={qs[0]:.3f}  Q1={qs[1]:.3f}  median={qs[2]:.3f}  Q3={qs[3]:.3f}  P95={qs[4]:.3f}  max={qs[5]:.3f}")


In [ ]:
# Cell 6 - 90% prediction intervals (computed in log space, inverted to $)
safe_std_log = np.clip(test_std_log, 1e-6, None)
lower_log = test_mean_log - 1.645 * safe_std_log
upper_log = test_mean_log + 1.645 * safe_std_log
pred_df = pd.DataFrame(
    {
        "actual": y_test_r.values,
        "mean":   test_mean,                       # already $/MWh
        "std_log": safe_std_log,                   # log-space (informational)
        "lower_90": np.expm1(lower_log),           # inverted to $/MWh
        "upper_90": np.expm1(upper_log),           # inverted to $/MWh
        "p_price_gt_200": spike_proba,
    },
    index=te_reg.index,
)

window = pred_df.loc["2024-01-13":"2024-01-20"].copy()
if window.empty:
    print("No rows found for the Jan 2024 cold snap window; choose another 2024 slice.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})

    axes[0].fill_between(
        window.index,
        window["lower_90"],
        window["upper_90"],
        color="lightsteelblue",
        alpha=0.45,
        label="90% prediction interval",
    )
    axes[0].plot(window.index, window["actual"], color="black", lw=1.2, label="Actual HB_HUBAVG")
    axes[0].plot(window.index, window["mean"], color="steelblue", lw=1.2, label="NGBoost mean")
    axes[0].axhline(200, color="firebrick", lw=0.9, linestyle="--", label="$200 spike threshold")
    axes[0].set_title("NGBoost 90% Prediction Intervals - Jan 2024 cold snap")
    axes[0].set_ylabel("$/MWh")
    axes[0].legend(loc="upper left", fontsize=9)

    axes[1].fill_between(window.index, window["p_price_gt_200"], color="firebrick", alpha=0.25)
    axes[1].plot(window.index, window["p_price_gt_200"], color="firebrick", lw=1.0, label="P(price > 200)")
    axes[1].axhline(0.5, color="gray", lw=0.8, linestyle="--", label="threshold=0.5")
    axes[1].set_ylabel("Probability")
    axes[1].set_ylim(0, 1)
    axes[1].legend(loc="upper left", fontsize=9)

    plt.tight_layout()
    plt.show()

    display(window[["actual", "mean", "std", "lower_90", "upper_90", "p_price_gt_200"]].head())


In [ ]:
from sklearn.metrics import average_precision_score
import numpy as np
_y_true_dollar = np.asarray(y_test_r).reshape(-1)
_y_pred_dollar = np.asarray(test_mean).reshape(-1)
_spike_pr_auc = average_precision_score(
    (_y_true_dollar > 200).astype(int),
    _y_pred_dollar,
)
print(f"\n=== Bake-off PR-AUC (regressor-as-score, threshold $200) ===")
print(f"  Spike PR-AUC: {_spike_pr_auc:.3f}")